# பாடம் 10 - உற்பத்தியில் AI முகவர்கள்

இந்த பாடத்தில் நீங்கள் Microsoft Agent Framework (Python) பயன்படுத்தி AI முகவர்களுக்கு **உற்பத்தி மாதிரிகள்** பற்றி கற்றுக்கொள்ளப்படுவீர்கள். நாங்கள் எடுத்துக் கொள்கிறோம்:

- **கண்காணிப்பு** — முகவர் தொடர்புகளில் நேரம் மற்றும் பதிவு சேர்த்தல்
- **மதிப்பீடு** — பதில் தரத்தை மதிப்பிடும் ஒரு மதிப்பீடு முகவரைப் பயன்படுத்துதல்
- **செலவு மேலாண்மை** — டோக்கன் சிறப்பித்தல் மற்றும் மாதிரி தேர்வு உத்திகள்

இந்த காட்சி ஒரு **பயண முகவர்** ஆகும், பயண திட்டமிட உதவுகிறது, கண்காணிப்பு மற்றும் மதிப்பீடு அதன் மேல் அடுக்கப்பட்டுள்ளது.


## அமைப்பு


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## உற்பத்தி கவனிப்புகள்

செயற்கை நுண்ணறிவு முகவர்கள் மாதிரிகளை இருந்து உற்பத்திக்கு நகர்வது மூன்று முக்கிய அஸ்திகளுக்கு நுட்பமான கவனத்தை தேவைப்படுத்துகிறது:

1. **கவனிப்புத்தன்மை** — முகவர் என்ன செய்கிறார், எவ்வளவு நேரம் எடுக்கிறது, மற்றும் எந்த கருவிகளைப் பயன்படுத்துகிறது என்பதைக் காண்பது அவசியம். பின்தொடர்பு மற்றும் பதிவு இல்லாமல், உற்பத்தி பிரச்சனைகளைக் கண்டுபிடிப்பது כמעט невозможным.

2. **மதிப்பீடு** — தானாக செய்யப்பட்ட தரமான சரிபார்ப்புகள் முகவரின் பதில்கள் சரியானவை, முழுமையானவை மற்றும் உதவிகளையும் காலக்கெடுவின்றி உறுதி செய்கிறது. மதிப்பீடு செயும் முகவர் வரையறுக்கப்பட்ட தரக்குறிப்புகளுக்கு எதிராக பதில்களை மதிப்பிடலாம்.

3. **செலவு மேலாண்மை** — டோக்கன் பயன்பாடு நேரடியாக செலவுக்கு தாக்கம் செலுத்துகிறது. தூண்டுகோள் மேம்பாடு, மாதிரி தேர்வு மற்றும் கேசிங் போன்ற முன்னேற்றங்கள் தரம் குறையாமல் செலவுகளை கட்டுப்பாட்டில் வைத்திருக்க உதவுகின்றன.


## ஒரு பார்வையிடக்கூடிய முகவரியை கட்டமைத்தல்

பயண கருவிகளை வரையறுக்கின்றோம் மற்றும் முகவர் அழைப்பை நேரத்தை கண்காணிக்க சுருட்டுகிறோம். உற்பத்தியில் நீங்கள் OpenTelemetry அல்லது அதே மாதிரியான தடயமாக்கல் பிரதியுடன் ஒருங்கிணைப்பீர்கள்.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## மதிப்பீடு முறைமைகள்

ஒரு பொதுவான உற்பத்தி முறைமையானது இரண்டாவது முகவரியை ஒரு **மதிப்பீட்டாளராக** பயன்படுத்துவது. மதிப்பீட்டாளர் முதன்மை முகவரியின் பதிலை முழுமையாகவும், துல்லியமாகவும், உதவிகரமாகவும் ஆகிய முன்கூட்டியே அமைக்கப்பட்ட критерியர்களுக்கு எதிராக மதிப்பிடுகிறார்.

இது செய்கின்றது:
- பதில்கள் பயனர்களுக்கு செல்லும் முன்னர் தானியங்கி தரம் வாயில்களை
- கோரிக்கைகள் அல்லது மாதிரிகள் மாற்றப்பட்டால் பின்னடைவுகளை கண்டறிதல்
- நேரத்தின் போக்கில் முகவரியின் செயல்திறனை நிலையான கண்காணிப்பு


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## செலவு மேலாண்மை நெறிமுறைகள்

தயாரிப்பு AI முகவர்களுக்கு செலவுகளை கட்டுப்படுத்துவது மிக முக்கியம். இங்கு முக்கியமான நெறிமுறைகள் உள்ளன:

| நெறிமுறை | விளக்கம் |
|---|---|
| **பிராம்ட் மேம்பாடு** | அமைப்பு নির্দেশங்களை சுருச்சடியாக வைத்திருக்கவும். உள்ளீட்டு டோக்கன்களை குறைக்க தேவையற்ற சூழ்நிலையை அகற்றவும். |
    "| **மாதிரி தேர்வு** | வகைப்படுத்தல் அல்லது சுருக்கம் போன்ற எளிய பணிகளுக்கு சிறிய, குறைந்த செலவு உள்ள மாதிரிகள் (எ.கா., GPT-4.1-mini) பயன்படுத்தவும், மற்றும் சிக்கலான கருத்துமுறை பணிகளுக்கு பெரிய மாதிரிகளை இடைநிறுத்தவும். |\n",
| **கேசிங்** | மீண்டும் மீண்டும் API அழைப்புகளை தவிர்க்க கருவி முடிவுகள் மற்றும் அடிக்கடி கேள்விகளை கேச் செய்யவும். |
| **டோக்கன் பட்ஜெட்டுகள்** | எதிர்பாராத நீண்ட பதில்களை தடுக்கும் வகையில் `max_tokens` வரம்புகளை அமைக்கவும். |
| **தொகுப்பு மாற்றம்** | சாத்தியமான இடத்தில் பல பயனர் கேள்விகளை ஒரே API அழைப்பாக கூட்டு செய்யவும். |

நடைமுறையில், ஒரு படிக்கட்டப்பட்ட அணுகுமுறை சிறந்தது: நேரடியாக எளிய கோரிக்கைகளை வேகமான மற்றும் குறைந்த செலவு உள்ள மாதிரிக்கு வழிமொழியவும், மற்றும் சிக்கலான கேள்விகள் மட்டுமே திறமையான மாதிரிக்கு உயர்த்தவும்.


## சுருக்கம்

இந்த பாடத்தில் நீங்கள் எப்படி என்பதை கற்றுக்கொண்டீர்கள்:

1. **கால அளவீடு மற்றும் பதிவு செய்யல் மூலம்** முகவர் தொடர்புகளில் பின்தண்ட யாக்கத்தையும் கண்காணிப்பையும் மேற்கொள்ளும் அடித்தளத்தை அமைத்தல்.
2. **முகவர் பதில்களை தானியங்கி மதிப்பீடு செய்வது**, முழுமையை, துல்லியத்தையும், உதவித்தன்மையையும் மதிப்பீடு செய்யும் மதிப்பீட்டாளர் முகவரை பயன்படுத்துதல்.
3. **செலவுகளை நிர்வகித்தல்** மூலம் முன்மொழிவு மேம்படுத்தல், மாதிரி தேர்வு, கேச்சிங் மற்றும் டோக்கன் பட்ஜெட்டுகள்.

இவை உங்களது AI முகவர்கள் நம்பகமான, அளவிடக்கூடிய மற்றும் செலவுக்குத் தகுந்ததாக இருக்க உதவும்.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
